In [ ]:
####################################
#ENVIRONMENT SETUP

In [ ]:
#LIBRARIES

#system
import os
os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"
import sys

#math and array operations
import numpy as np
import pandas as pd
import math

#plotting
import matplotlib
# matplotlib.use("Agg") #UNCOMMENT IF PLOTTING WITHIN JUPYTER DOCUMENT
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import TwoSlopeNorm
from matplotlib.colors import ListedColormap, BoundaryNorm

import cartopy.crs as ccrs
import cartopy.feature as cfeature

#data classes
import xarray as xr
import h5py
import pickle 

#loading bar
from tqdm import tqdm

#dates
from datetime import datetime, timedelta

In [ ]:
#Importing DirectoryManager Class
sys.path.append(os.path.join("/glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/CodeFiles/","DataAnalysis"))
from CLASSES_Directories import DirectoryManager_Class

In [ ]:
DirectoryManager = DirectoryManager_Class()

codeType = os.path.join("DataAnalysis", "Observation_Data")
dataType = "RainfallHistogram"

outputDirectory = DirectoryManager.GetOutputDirectory(codeType, dataType)
outputPlottingDirectory = DirectoryManager.GetOutputPlottingDirectory(codeType, dataType)

In [ ]:
#Importing ModelData Class
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis","MPAS_Model_Data"))
from CLASSES_ModelData import StructuredModelData_Class, DataOperator_Class

In [ ]:
#Importing ModelData Class
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis","MPAS_Model_Data"))
from CLASSES_PlottingModelData import FigurePlotting_Class

In [ ]:
#Setup

Region = "PRECIP"; Case = "WET"; spinup_hours = "12"
Region = "PRECIP"; Case = "DIURNAL"; spinup_hours = "12"

In [ ]:
#Load Model Directory Class
RunType = (Region,Case,"NSSL",spinup_hours)
ModelData_NSSL = StructuredModelData_Class(DirectoryManager.mainDirectory, DirectoryManager.scratchDirectory, RunType)

RunType = (Region,Case,"TEMPO",spinup_hours)
ModelData_TEMPO = StructuredModelData_Class(DirectoryManager.mainDirectory, DirectoryManager.scratchDirectory, RunType)

In [ ]:
#Importing ModelData Class
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis"))
from CLASSES_DataSubsetting import DataSubsetting_Class

In [ ]:
#Importing Radar Classes
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis","Observation_Data"))
from CLASSES_RadarDataLoading import RadarData_MRMS_Class, RadarObservationMask_Class

sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis"))
from CLASSES_RadarDataPlotting import RadarPlotting_Class

In [ ]:
#IMPORT FUNCTIONS
# --- Add your Functions folder to sys.path ---
import sys
path = os.path.join(DirectoryManager.mainCodeDirectory, 'Functions_2.0')
sys.path.append(path)


# --- Import all your function modules ---
import importlib
modules = [
    "AreaAverageFunctions",
    "ComputationFunctions",
    "DataFunctions",
    "DerivativeFunctions",
    "PlottingFunctions",
    "StatisticalFunctions",
]
for mod in modules:
    globals()[mod] = importlib.import_module(mod)        # import module itself
    globals().update(vars(globals()[mod]))              # import all functions into global namespace

In [ ]:
####################################
#GETTING PRECIP DATA

In [ ]:
def SubsetData_Time(data,yearmonthday):
    data_T = data.sel(time=yearmonthday)
    # print(data_T.time) #testing
    return data_T

In [ ]:
# #TESTING LOADING AND PLOTTING

# #LOADING DATA
# import struct
# filePath = "CB_GC_PCP_1H_RAD.20220606.0400"
# Ny = 441; Nx = 561
# f=open(filePath,'rb')
# header=f.read(170) # header
# rain=np.reshape(struct.unpack('@'+str(Ny*Nx)+'h',f.read(Ny*Nx*2)),[Ny,Nx],'F')
# rain = rain / 4 #converting from units of 0.25 mm to mm
# rain[rain<0] = np.nan
# rain = rain.T
# rain = rain[::-1]

# #GETTING COORDINATES
# def BuildQPELonLat(nx, ny):
#     dx = 12.5 / 1000.0
#     lon0 = 118.0
#     lat0 = 27.0

#     lon = lon0 + dx * np.arange(nx)
#     lat = lat0 - dx * np.arange(ny)

#     qpeLon, qpeLat = np.meshgrid(lon, lat)
#     return qpeLon, qpeLat
# ny, nx = rain.shape
# qpeLon, qpeLat = BuildQPELonLat(nx, ny)

# #####################################

# fig = plt.figure(figsize=(10, 8))

# ax = plt.axes(projection=ccrs.PlateCarree())

# cf = ax.contourf(
#     qpeLon,
#     qpeLat,
#     rain,
#     levels=20,
#     transform=ccrs.PlateCarree(),
#     cmap="turbo"
# )

# # --- Map features ---
# ax.coastlines(resolution="10m", linewidth=1.2)
# ax.add_feature(cfeature.BORDERS, linewidth=1.0)
# ax.add_feature(cfeature.LAND, facecolor="none", edgecolor="black", linewidth=0.5)
# ax.add_feature(cfeature.OCEAN, facecolor="none")

# # Optional: Taiwan outline looks better with this
# ax.add_feature(cfeature.STATES, linewidth=0.5)

# # --- Domain bounds (important!) ---
# ax.set_extent(
#     [qpeLon.min(), qpeLon.max(), qpeLat.min(), qpeLat.max()],
#     crs=ccrs.PlateCarree()
# )
# gl = ax.gridlines(
#     crs=ccrs.PlateCarree(),
#     draw_labels=True,
#     linewidth=0.8,
#     color="gray",
#     alpha=0.6,
#     linestyle="--"
# )
# gl.top_labels = False
# gl.right_labels = False

# plt.colorbar(cf, ax=ax, label="Rainfall (mm)")
# fig.suptitle("Accumulated rainfall at 20220606.0400")

In [ ]:
#DATA LOADING FOR PRECIP QPE DATA

def CorrectSimulationDates(ModelData, simulationDates):
    if int(ModelData.spinup_hours) <= 0:
        # Convert first date to Timestamp
        first_date = pd.to_datetime(simulationDates[0])
        # Subtract one day
        prev_date = (first_date - pd.Timedelta(days=1)).strftime('%Y-%m-%d')
        # Prepend
        simulationDates = [prev_date] + simulationDates
    return simulationDates

def GetPRECIP_QPE_DataDirectory(ModelData):
    simulationDates = ModelData.simulationDates
    simulationDates2 = CorrectSimulationDates(ModelData, simulationDates)

    inputDirectory = os.path.join(DirectoryManager.dataDirectory,
             f"Observation_Data/PRECIP/QPE1HR", ModelData.case)
    return inputDirectory

import struct

def BuildQPELonLat(nx, ny):
    dx = 12.5 / 1000.0
    lon0 = 118.0
    lat0 = 27.0

    lon = lon0 + dx * np.arange(nx)
    lat = lat0 - dx * np.arange(ny)

    qpeLon, qpeLat = np.meshgrid(lon, lat)
    return qpeLon, qpeLat


def ReadQPEData(filePath):
    f=open(filePath,'rb')
    header=f.read(170) # header
    rain=np.reshape(struct.unpack('@'+str(441*561)+'h',f.read(441*561*2)),[441,561],'F')
    rain = rain / 4 #converting from units of 0.25 mm to mm
    rain[rain<0] = np.nan
    rain = rain.T
    rain = rain[::-1]

    ny,nx=rain.shape
    return rain, nx,ny 

# def GetPreviousHourQPEFiles(yearmonthdayhour, inputDirectory):
#     """
#     yearmonthdayhour : 'YYYYMMDDHH'
#     returns list of full file paths covering the previous hour
#     """

#     targetTime = pd.to_datetime(yearmonthdayhour, format="%Y%m%d%H")
#     startTime  = targetTime - pd.Timedelta(hours=1)

#     yearmonthday = targetTime.strftime("%Y%m%d")
#     inputFolder = os.path.join(inputDirectory, yearmonthday)

#     fileList = []

#     currentTime = startTime + pd.Timedelta(minutes=10)
#     while currentTime <= targetTime:
#         timestamp = currentTime.strftime("%H%M")
#         fileName = f"CB_GC_PCP_1H_RAD.{yearmonthday}.{timestamp}.gz"
#         fullPath = os.path.join(inputFolder, fileName)

#         if os.path.exists(fullPath):
#             fileList.append(fullPath)

#         currentTime += pd.Timedelta(minutes=10)

#     return sorted(fileList)
# fileList = GetPreviousHourQPEFiles(yearmonthdayhour, inputDirectory) #incorrect since each 10 minute file is accumulation of past horu

def ReadPrecipData_PRECIP_QPE(ModelData,yearmonthdayhour,inputDirectory):
    yearmonthday = yearmonthdayhour[0:8]
    hour = yearmonthdayhour[8:]
    inputFolder = os.path.join(inputDirectory,yearmonthday)
    filePath = os.path.join(inputFolder, f"CB_GC_PCP_1H_RAD.{yearmonthday}.{hour}00")#.gz")
    # print(filePath) #* testing

    precipData,qpeLon,qpeLat = ReadQPEData(filePath)
    return precipData, qpeLon,qpeLat


def Plot_PRECIP_QPE(ModelData): 
    #getting inputDirectory
    inputDirectory = GetPRECIP_QPE_DataDirectory(ModelData)
    print(f"reading from {inputDirectory}")

    #getting yearmonthdayhour list
    dt_list = [datetime.strptime(t, "%Y-%m-%d_%H.%M.%S") for t in ModelData.timeStrings]
    start = dt_list[0]
    target = start + timedelta(hours=13)
    # print(f"start_time: {target}")
    
    # Find the entry closest to +12 hours
    closest = min(dt_list, key=lambda x: abs(x - target))
    
    closest_idx = dt_list.index(closest)
    times = ModelData.timeStrings[closest_idx:]
    yearmonthdayhours = sorted({t.replace('-', '').replace('_', '').replace('.', '')[:10] 
                                for t in times})

    precipData_T = None
    for count, yearmonthdayhour in tqdm(enumerate(yearmonthdayhours), total=len(yearmonthdayhours)):
        # print(yearmonthdayhour)
        
        precipData,nx,ny = ReadPrecipData_PRECIP_QPE(ModelData,yearmonthdayhour,inputDirectory)
        if count == 0:
            qpeLon,qpeLat = BuildQPELonLat(nx,ny)

        if precipData is None:
            continue #skips missing hours
        if precipData_T is None:
            precipData_T = precipData #equates at first timestep
        else:
            precipData_T += precipData

        # 👇 plot every 10 steps
        if count % 5 != 0:
            continue

        plt.figure(figsize=(8, 6))
        cf = plt.contourf(qpeLon, qpeLat, precipData_T, levels=20)
        plt.colorbar(cf, label="QPE (mm)")
        plt.title(yearmonthdayhour)
        plt.show()
# Plot_PRECIP_QPE(ModelData_NSSL)  


def GetAccumulatedPrecipData_PRECIP_QPE(ModelData): 
    #getting inputDirectory
    inputDirectory = GetPRECIP_QPE_DataDirectory(ModelData)
    print(f"reading from {inputDirectory}")

    #getting yearmonthdayhour list
    dt_list = [datetime.strptime(t, "%Y-%m-%d_%H.%M.%S") for t in ModelData.timeStrings]
    start = dt_list[0]
    target = start + timedelta(hours=13)
    # print(f"start_time: {target}")
    
    # Find the entry closest to +12 hours
    closest = min(dt_list, key=lambda x: abs(x - target))
    
    closest_idx = dt_list.index(closest)
    times = ModelData.timeStrings[closest_idx:]
    yearmonthdayhours = sorted({t.replace('-', '').replace('_', '').replace('.', '')[:10] 
                                for t in times})

    precipData_T = None
    for count, yearmonthdayhour in tqdm(enumerate(yearmonthdayhours), total=len(yearmonthdayhours)):
        # print(yearmonthdayhour)
        
        precipData,nx,ny = ReadPrecipData_PRECIP_QPE(ModelData,yearmonthdayhour,inputDirectory)
        if precipData is None:
            continue #skips missing hours
        if precipData_T is None:
            precipData_T = precipData #equates at first timestep
        else:
            precipData_T += precipData

    qpeLon,qpeLat = BuildQPELonLat(nx,ny)
    
    precipData_T_xarray = xr.DataArray(
        precipData_T,              # or rain
        dims=("latitude", "longitude"),
        coords={
            "longitude": qpeLon[0, :],
            "latitude": qpeLat[:, 0]
        },
        name="PRECIP_QPE"
    )
    return precipData_T_xarray

def LoadOrRunGetAccumulatedPrecipData_PRECIP_QPE(
    ModelData,
    overwrite=False):
    """
    Load accumulated PRECIP QPE precipitation from disk if available,
    otherwise compute it using GetAccumulatedPrecipData_PRECIP_QPE.
    """

    os.makedirs(outputDirectory, exist_ok=True)

    fileName = (
        f"{ModelData.region}_"
        f"{ModelData.case}_"
        f"{ModelData.spinup_hours}hrs_"
        f"PRECIP_QPE_accumulated.nc"
    )
    filePath = os.path.join(outputDirectory, fileName)

    # --------------------------------------------------
    # Load
    # --------------------------------------------------
    if os.path.exists(filePath) and not overwrite:
        print(f"Loading PRECIP QPE accumulated precip from disk:")
        print(f"  {filePath}")
        with xr.open_dataset(filePath) as ds:
            precipData_T = ds["PRECIP_QPE"].load()
        return precipData_T

    # --------------------------------------------------
    # Run
    # --------------------------------------------------
    print("Computing PRECIP QPE accumulated precipitation")
    precipData_T = GetAccumulatedPrecipData_PRECIP_QPE(ModelData)

    # --------------------------------------------------
    # Save
    # --------------------------------------------------
    precipData_T.to_dataset(name="PRECIP_QPE").to_netcdf(filePath)
    print(f"Saved PRECIP QPE accumulated precip to disk:")
    print(f"  {filePath}")

    return precipData_T


In [ ]:
precipData_T = LoadOrRunGetAccumulatedPrecipData_PRECIP_QPE(ModelData_NSSL)
precipData_T_Subset = DataSubsetting_Class.SubsetDataRegion(precipData_T, ModelData_NSSL)

In [ ]:
####################################
#CALCULATION FUNCTIONS

In [ ]:
def GetVariableSubset_Helper(ModelData, dataSubset, dataSubset_diag, dataSubset_static, varName):
    """
    Retrieves a variable subset from the given model data.
    If varName contains a '+', returns the sum of the two variables.
    """
    if '+' in varName:
        var1, var2 = varName.split('+')
        var1 = var1.strip()
        var2 = var2.strip()

        subset1 = DataOperator_Class.GetData_Variable(ModelData, dataSubset, 
                                                      dataSubset_diag, dataSubset_static, var1)
        subset2 = DataOperator_Class.GetData_Variable(ModelData, dataSubset, 
                                                      dataSubset_diag, dataSubset_static, var2)
        variableSubset = subset1 + subset2
    else:
        variableSubset = DataOperator_Class.GetData_Variable(ModelData, dataSubset, 
                                                             dataSubset_diag, dataSubset_static, varName)

    return variableSubset

# def RunCalculations(ModelData, varName):
#     #Gets the last timestep rainnc+rainc and later RemoveFirstTwelveHours removes the spinup rain
#     #(Another option is to sum raincv, rainncv separately)
#     outputDictionary={}
    
#     t = ModelData.Ntime-1
        
#     #Loading Data
#     [dataSubset, dataSubset_diag, dataSubset_static, lat, lon, _,_, _,_] = DataOperator_Class.GetData_Subset(ModelData, t)


#     print(f"Running for {varName}")
#     #Subsetting Data

#     variableSubset = GetVariableSubset_Helper(ModelData, dataSubset, dataSubset_diag, dataSubset_static, varName)
        
#     return variableSubset, variableSubset.latitude, variableSubset.longitude

def RunCalculations(ModelData, varName, 
                    overwrite=False):
    """
    Load precomputed variable from disk if available,
    otherwise calculate and save it.
    """
    
    os.makedirs(outputDirectory, exist_ok=True)

    fileName = f"{ModelData.region}_{ModelData.case}_{ModelData.spinup_hours}hrs_{ModelData.mpType}_{varName}.nc"
    filePath = os.path.join(outputDirectory, fileName)

    # --------------------------------------------------
    # Load from disk if it exists
    # --------------------------------------------------
    if os.path.exists(filePath) and not overwrite:
        print(f"Loading from disk: {filePath}")
        variableSubset = xr.open_dataset(filePath)[varName]
        return variableSubset, variableSubset.latitude, variableSubset.longitude

    # --------------------------------------------------
    # Otherwise compute
    # --------------------------------------------------
    print(f"Running calculation for {varName}")

    t = ModelData.Ntime - 1

    [dataSubset,
     dataSubset_diag,
     dataSubset_static,
     lat,
     lon,
     _, _, _, _] = DataOperator_Class.GetData_Subset(ModelData, t)

    variableSubset = GetVariableSubset_Helper(
        ModelData,
        dataSubset,
        dataSubset_diag,
        dataSubset_static,
        varName
    )

    # --------------------------------------------------
    # Save to disk
    # --------------------------------------------------
    variableSubset.to_dataset(name=varName).to_netcdf(filePath)
    print(f"Saved to disk: {filePath}")

    return variableSubset, variableSubset.latitude, variableSubset.longitude


In [ ]:
def FindIndexAtHourOffset(timeStrings, hourOffset):
    # Convert strings → datetime objects
    dt_list = [datetime.strptime(t, "%Y-%m-%d_%H.%M.%S") for t in timeStrings]
    
    # Starting time
    start = dt_list[0]
    target = start + timedelta(hours=hourOffset)
    
    # Find closest timestamp
    closest_dt = min(dt_list, key=lambda x: abs(x - target))
    idx = dt_list.index(closest_dt)
    
    return idx, timeStrings[idx]
    
def RemoveFirstTwelveHours(ModelData, rain_model):
    varName = 'rainnc+rainc'
    t12, _ = FindIndexAtHourOffset(ModelData.timeStrings, 12)
    [dataSubset, dataSubset_diag, dataSubset_static, lat, lon, _,_,  _,_] = DataOperator_Class.GetData_Subset(ModelData, t12)
    variableSubset = GetVariableSubset_Helper(ModelData, dataSubset, dataSubset_diag, dataSubset_static, varName)
    rain_model -= variableSubset
    return rain_model

In [ ]:
####################################
#CALCULATION

In [ ]:
rain_NSSL, lat,lon = RunCalculations(ModelData=ModelData_NSSL,varName='rainnc+rainc')
rain_TEMPO, _,_ = RunCalculations(ModelData=ModelData_TEMPO,varName='rainnc+rainc')

In [ ]:
rain_NSSL = RemoveFirstTwelveHours(ModelData=ModelData_NSSL, rain_model=rain_NSSL)
rain_TEMPO = RemoveFirstTwelveHours(ModelData=ModelData_TEMPO, rain_model=rain_TEMPO)

In [ ]:
# #Applying Radar Mask #decided not use here
# RadarDataMask = RadarObservationMask_Class.LoadMaskData_MRMS(DirectoryManager, ModelData_NSSL)
# RadarDataMask_interp = (RadarDataMask*1).interp(
#     latitude = precipData_T_Subset.latitude,
#     longitude = precipData_T_Subset.longitude,
#     method="nearest"
# )
# RadarDataMask_interp = RadarDataMask_interp == 1

# rain_NSSL = rain_NSSL.where(RadarDataMask == True) #commenting out since precip data has wider mask
# rain_TEMPO = rain_TEMPO.where(RadarDataMask == True)
# precipData_T_Subset = precipData_T_Subset.where(RadarDataMask_interp == True)

In [ ]:
# def InterpToStageIV(model_var, model_lat, model_lon, stage_lat2d, stage_lon2d): 
#     """
#     Interpolates MPAS (lat1d, lon1d) field to Stage-IV curvilinear grid.

#     Returns DataArray with EXACT Stage-IV dims/coords
#     (no artificial y/x dims).
#     """

#     # 1. Extract numpy values
#     model_array = np.asarray(model_var)

#     # 2. Build MPAS points
#     mpas_lon2d, mpas_lat2d = np.meshgrid(model_lon, model_lat)
#     points = np.column_stack([mpas_lon2d.ravel(), mpas_lat2d.ravel()])
#     values = model_array.ravel()

#     # 3. Target grid (Stage-IV)
#     stage_points = np.column_stack([stage_lon2d.ravel(), stage_lat2d.ravel()])

#     # 4. Interpolate
#     interp_flat = griddata(points, values, stage_points, method="linear")
#     interp2d = interp_flat.reshape(stage_lat2d.shape)

#     # 5. Return DataArray preserving Stage-IV coords
#     da = xr.DataArray(
#         interp2d,
#         dims=precipData_T_Subset.dims,  # <<— IMPORTANT
#         coords={
#             "latitude": (precipData_T_Subset.dims, stage_lat2d),
#             "longitude": (precipData_T_Subset.dims, stage_lon2d)
#         },
#         name=model_var.name if hasattr(model_var, "name") else "interpolated"
#     )

#     return da


# rain_NSSL_interp = InterpToStageIV(
#     rain_NSSL, lat, lon,
#     precipData_T_Subset.latitude.values,
#     precipData_T_Subset.longitude.values
# )

# rain_TEMPO_interp = InterpToStageIV(
#     rain_TEMPO, lat, lon,
#     precipData_T_Subset.latitude.values,
#     precipData_T_Subset.longitude.values
# )


In [ ]:
#Interpolatting model data onto MRMS data
rain_NSSL_interp = rain_NSSL.interp(longitude=precipData_T_Subset.longitude,latitude=precipData_T_Subset.latitude)
rain_TEMPO_interp = rain_TEMPO.interp(longitude=precipData_T_Subset.longitude,latitude=precipData_T_Subset.latitude)
clim = (
    min(rain_NSSL_interp.min().item(), 
        rain_TEMPO_interp.min().item(), 
        precipData_T_Subset.min().item()),

    max(rain_NSSL_interp.max().item(), 
        rain_TEMPO_interp.max().item(), 
        precipData_T_Subset.max().item())
)

In [ ]:
#Removing Model Regions with NaN in Observations
rain_NSSL_interp = rain_NSSL_interp.where(~np.isnan(precipData_T_Subset))
rain_TEMPO_interp = rain_TEMPO_interp.where(~np.isnan(precipData_T_Subset))

In [ ]:
#coloring zeros white (doesn't effect histograms)

rain_NSSL_interp_whited = rain_NSSL_interp.where(rain_NSSL_interp!=0)
rain_TEMPO_interp_whited = rain_TEMPO_interp.where(rain_TEMPO_interp!=0)
precipData_T_Subset_whited = precipData_T_Subset.where(precipData_T_Subset!=0)

#another option
# rain_NSSL_interp_whited = rain_NSSL_interp.where(rain_NSSL_interp>=0.1)
# rain_TEMPO_interp_whited = rain_TEMPO_interp.where(rain_TEMPO_interp>=0.1)
# precipData_T_Subset_whited = precipData_T_Subset.where(precipData_T_Subset>=0.1)

In [ ]:
def CreateFigure_2x2(figsize=(10, 10),
                     wspace=0.3, hspace=0.3,
                     left=0.05, right=0.95, top=0.92, bottom=0.08):
    """
    Creates a 2x2 grid of subplots with adjustable layout.
    Returns (fig, axes) where axes is a 2D list [[ax00, ax01], [ax10, ax11]].
    """
    fig = plt.figure(figsize=figsize)
    fig.subplots_adjust(left=left, right=right, top=top, bottom=bottom,
                        wspace=wspace, hspace=hspace)

    gs = gridspec.GridSpec(2, 2, figure=fig)

    ax00 = fig.add_subplot(gs[0, 0], projection=ccrs.PlateCarree())
    ax01 = fig.add_subplot(gs[0, 1], projection=ccrs.PlateCarree())
    ax10 = fig.add_subplot(gs[1, 0])
    ax11 = fig.add_subplot(gs[1, 1])

    axes = [[ax00, ax01],
            [ax10, ax11]]

    return fig, axes

def CreateFigure_2x1Combo(figsize=(10, 10),
                          wspace=0.35, hspace=0.1,
                          left=0.07, right=0.95, top=0.92, bottom=0.08):
    """
    Creates a figure with two subplots on top (map style)
    and one wide subplot spanning the full bottom row.
    
    Layout:
        +-----------+-----------+
        |   ax00    |   ax01    |
        +-----------------------+
        |        ax_bottom      |
        +-----------------------+
    
    Returns (fig, axes) where:
      axes = [[ax00, ax01], [ax_bottom]]
    """
    fig = plt.figure(figsize=figsize)
    fig.subplots_adjust(left=left, right=right, top=top, bottom=bottom,
                        wspace=wspace, hspace=hspace)

    gs = gridspec.GridSpec(2, 2, figure=fig, height_ratios=[1, 0.8])

    # Two map panels on top
    ax00 = fig.add_subplot(gs[0, 0], projection=ccrs.PlateCarree())
    ax01 = fig.add_subplot(gs[0, 1], projection=ccrs.PlateCarree())

    # One wide histogram panel across bottom
    ax_bottom = fig.add_subplot(gs[1, :])  # spans both columns

    axes = [[ax00, ax01], [ax_bottom]]

    return fig, axes

# def CreateFigure_3x1(figsize=(14, 10),
#                      wspace=0.05, hspace=0.15,
#                      left=0.06, right=0.97, top=0.93, bottom=0.08):
#     """
#     Layout:
#         +-----------+-----------+-----------+
#         |   ax00    |   ax01    |   ax02    |
#         +-----------------------------------+

#     Returns (fig, axes) where:
#       axes = [[ax00, ax01, ax02]]
#     """

#     fig = plt.figure(figsize=figsize)
#     fig.subplots_adjust(left=left, right=right, top=top, bottom=bottom,
#                         wspace=wspace, hspace=hspace)

#     # 2 rows × 3 columns grid
#     gs = gridspec.GridSpec(1, 3, figure=fig)

#     # Top row: 3 map panels
#     ax00 = fig.add_subplot(gs[0, 0], projection=ccrs.PlateCarree())
#     ax01 = fig.add_subplot(gs[0, 1], projection=ccrs.PlateCarree())
#     ax02 = fig.add_subplot(gs[0, 2], projection=ccrs.PlateCarree())

#     axes = [[ax00, ax01, ax02]]

#     return fig, axes

def CreateFigure_3x1(figsize=(14, 10),
                     wspace=0.05, hspace=0.15,
                     left=0.06, right=0.97, top=0.93, bottom=0.08):

    fig = plt.figure(figsize=figsize)

    # 3 panels + 1 colorbar column
    gs = gridspec.GridSpec(
        1, 4,
        figure=fig,
        width_ratios=[1, 1, 1, 0.05],  # last column = colorbar
        wspace=wspace
    )

    ax00 = fig.add_subplot(gs[0, 0], projection=ccrs.PlateCarree())
    ax01 = fig.add_subplot(gs[0, 1], projection=ccrs.PlateCarree())
    ax02 = fig.add_subplot(gs[0, 2], projection=ccrs.PlateCarree())

    cax  = fig.add_subplot(gs[0, 3])  # colorbar axis

    axes = [[ax00, ax01, ax02]]
    return fig, axes, cax

def CreateFigure_1x1(figsize=(14, 10),
                     wspace=0.25, hspace=0.15,
                     left=0.06, right=0.97, top=0.93, bottom=0.08):

    fig = plt.figure(figsize=figsize)
    fig.subplots_adjust(left=left, right=right,
                        top=top, bottom=bottom,
                        wspace=wspace, hspace=hspace)

    gs = gridspec.GridSpec(1, 1, figure=fig)

    ax00 = fig.add_subplot(gs[0, 0])

    axes = [[ax00]]
    return fig, axes

In [ ]:
#CONTOUR PLOTTING FUNCTION
from mpl_toolkits.axes_grid1 import make_axes_locatable
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter

# Preload map features once (global)
COAST = cfeature.COASTLINE.with_scale("50m")
BORDERS = cfeature.BORDERS.with_scale("50m")
STATES = cfeature.STATES.with_scale("50m")
LAND = cfeature.LAND.with_scale("50m")
LAKES = cfeature.LAKES.with_scale("50m")

def BuildDiscreteColormap(levels, cmap="turbo"):
    cmap_obj = plt.get_cmap(cmap)
    # one color per level interval
    colors = cmap_obj(np.linspace(0, 1, len(levels)-1))
    discrete_cmap = ListedColormap(colors)
    norm = BoundaryNorm(levels, discrete_cmap.N, clip=False)
    return discrete_cmap, norm
def PlotVariable_with_Borders(axis, 
                              variable, varName, lat, lon, multiplier=1, 
                              clim=(None, None), norm=None, cmap="turbo",
                              title=None, units=None,
                              center_colorbar=False, levels=None,
                              showLat=True,showLon=True):

    # --- Compute bounds ---
    vmin = np.nanmin(variable) if clim[0] is None else clim[0]
    vmax = np.nanmax(variable) if clim[1] is None else clim[1]

    # --- Levels ---
    if levels is None:
        num_levels = 19
        levels = np.linspace(vmin, vmax, num_levels)
    else:
        levels = np.asarray(levels)

    matrix = multiplier * variable

    # --- Build discrete colormap + norm ---
    if norm is None:
        discrete_cmap, norm = BuildDiscreteColormap(levels, cmap=cmap)
    else:
         discrete_cmap = plt.get_cmap(cmap)

    # --- DISCRETE PCOLORMESH ---
    im = axis.pcolormesh(
        lon, lat, matrix,
        cmap=discrete_cmap,
        norm=norm,
        shading="nearest",
        transform=ccrs.PlateCarree()
    )

    # --- Map features ---
    axis.add_feature(COAST,  linewidth=1.2, edgecolor="green")
    axis.add_feature(BORDERS, linewidth=0.8, edgecolor="green")
    axis.add_feature(STATES,  linewidth=0.5, edgecolor="green")
    axis.add_feature(LAND, facecolor="lightgray", alpha=0.3)
    axis.add_feature(LAKES, edgecolor="k", facecolor="none")

    # --- Labels / axes ---
    axis.set_extent([lon.min(), lon.max(), lat.min(), lat.max()],
                    crs=ccrs.PlateCarree())
    # axis.set_xticks(np.round(np.linspace(lon.min(), lon.max(), 5), 1), #old
    #                 crs=ccrs.PlateCarree())
    # axis.set_yticks(np.round(np.linspace(lat.min(), lat.max(), 5), 1), #old
    #                 crs=ccrs.PlateCarree())
    gl = axis.gridlines(draw_labels=True, color='gray', alpha=1, linestyle='--')
    gl.top_labels = False
    gl.right_labels = False
    gl.left_labels = showLat     # show latitude
    gl.bottom_labels = showLon   # show longitude
    gl.xlabel_style = {'size': fontSettings["tickFont"]}
    gl.ylabel_style = {'size': fontSettings["tickFont"]}

    # axis.set_xlabel("Longitude (°E)")
    # axis.set_ylabel("Latitude (°N)")

    if title:
        axis.set_title(title)

    return im

In [ ]:
###################
#PLOTTING FUNCTIONS
fontSettings = {
    "tickFont": 16+2,
    "labelFont": 18+2,
    "legendFont": 14+2,
    "titleFont": 22,
}

In [ ]:
def PlotRainContours(axes, cax, rain_NSSL, rain_TEMPO, lat, lon, clim,
                     ModelData_NSSL, ModelData_TEMPO,
                     precipData_T_Subset,
                     num_levels=None,
                     logscale=True):
    """
    Plot spatial rain accumulation maps for NSSL, Stage-IV, and TEMPO,
    all using the MPAS model lat/lon limits.
    """

    #Removing Low Data
    rain_NSSL=rain_NSSL.where(rain_NSSL>=0.3)
    rain_TEMPO=rain_TEMPO.where(rain_TEMPO>=0.3)
    precipData_T_Subset=precipData_T_Subset.where(precipData_T_Subset>=0.3)

    # Shared discrete levels for all 3 panels
    if num_levels is None:
        num_levels = 19
    if logscale==True:
        shared_levels = np.logspace(np.log10(max(clim[0],0.1)), np.log10(clim[1]), num_levels)
    else:
        shared_levels = np.linspace(clim[0], clim[1], num_levels)

    # Domain limits
    # lat_min, lat_max = float(lat.min()), float(lat.max())
    # lon_min, lon_max = float(lon.min()), float(lon.max())
    lat_min = float(ModelData_NSSL.latitude.min())
    lat_max = float(ModelData_NSSL.latitude.max())
    lon_min = float(ModelData_NSSL.longitude.min())
    lon_max = float(ModelData_NSSL.longitude.max())
    
    # ------------------------------------------
    # 1. NSSL panel
    # ------------------------------------------
    axis = axes[0][0]
    im = PlotVariable_with_Borders(
        axis,
        rain_NSSL,
        "",
        lat, lon,
        clim=clim,
        levels=shared_levels
    )
    axis.set_title(f'NSSL',fontsize=fontSettings["labelFont"])#f'{ModelData_NSSL.region}_{ModelData_NSSL.case}_{ModelData_NSSL.mpType} rainnc+rainc')
    axis.set_xlim(lon_min, lon_max)
    axis.set_ylim(lat_min, lat_max)
    # axis.set_ylabel("longitude",fontsize=fontSettings["labelFont"])
    # axis.set_xlabel("latitude",fontsize=fontSettings["labelFont"])
    axis.tick_params(axis="both", labelsize=fontSettings["tickFont"])

    # ------------------------------------------
    # 2. Stage IV panel (now using same function)
    # ------------------------------------------
    axis = axes[0][1]
    PlotVariable_with_Borders(
        axis,
        precipData_T_Subset.values,
        "",
        precipData_T_Subset.latitude.values,
        precipData_T_Subset.longitude.values,
        clim=clim,
        levels=shared_levels,
        showLat=False
    )
    axis.set_title(f'PRECIP QPE',fontsize=fontSettings["labelFont"])# 01H')
    axis.set_xlim(lon_min, lon_max)
    axis.set_ylim(lat_min, lat_max)
    # axis.set_xlabel("latitude",fontsize=fontSettings["labelFont"])
    # axis.set_ylabel("",fontsize=fontSettings["labelFont"])
    axis.tick_params(axis="both", labelsize=fontSettings["tickFont"])

    # ------------------------------------------
    # 3. TEMPO panel
    # ------------------------------------------
    axis = axes[0][2]
    PlotVariable_with_Borders(
        axis,
        rain_TEMPO,
        "",
        lat, lon,
        clim=clim,
        levels=shared_levels,
        showLat=False
    )
    axis.set_title(f'TEMPO',fontsize=fontSettings["labelFont"])#f'{ModelData_TEMPO.region}_{ModelData_TEMPO.case}_{ModelData_TEMPO.mpType} rainnc+rainc')
    axis.set_xlim(lon_min, lon_max)
    axis.set_ylim(lat_min, lat_max)
    # axis.set_xlabel("latitude",fontsize=fontSettings["labelFont"])
    # axis.set_ylabel("",fontsize=fontSettings["labelFont"])
    axis.tick_params(axis="both", labelsize=fontSettings["tickFont"])

    # ------------------------------------------
    # 4. Shared colorbar
    # ------------------------------------------
    # cbar = fig.colorbar(
    #     im,
    #     ax=axes[0],         # attach to all first-row axes
    #     orientation="vertical",
    #     fraction=0.03,
    #     pad=0.02
    # )
    #####
    cbar = plt.colorbar(im, cax=cax)
    pos = axes[0][2].get_position()
    cax.set_position([
        cax.get_position().x0,
        pos.y0,
        cax.get_position().width,
        pos.height
    ])
    #####
    cbar.set_label("Rainfall (mm)",fontsize=fontSettings["labelFont"])
    cbar.set_ticks(shared_levels)
    
    from matplotlib.ticker import FuncFormatter
    
    cbar.formatter = FuncFormatter(lambda x, pos: f"{x:.2f}")
    cbar.update_ticks()

    cbar.ax.tick_params(axis="both", labelsize=fontSettings["tickFont"])

In [ ]:
def PlotRainHistograms(axes, rain_NSSL, rain_TEMPO, precipData_T_Subset,
                       axis=None):
    """
    Plot outlined histograms of accumulated precipitation for NSSL and TEMPO
    on a single combined bottom axis using shared bins and different colors.
    """
    if axis is None:
        # Define the bottom axis (only one now)
        axis = axes[1][0]

    # Getting valid non-nan data
    valid_NSSL  = rain_NSSL.values.flatten()
    valid_NSSL  = valid_NSSL[~np.isnan(valid_NSSL)]
    
    valid_TEMPO = rain_TEMPO.values.flatten()
    valid_TEMPO = valid_TEMPO[~np.isnan(valid_TEMPO)]
    
    valid_NOAA  = precipData_T_Subset.data.flatten()
    valid_NOAA  = valid_NOAA[~np.isnan(valid_NOAA)]

    # Common bins for fair comparison
    combinedData = np.concatenate([valid_NSSL, valid_TEMPO, valid_NOAA])
    binEdges = np.linspace(combinedData.min(), combinedData.max(), 51)  # 50 bins

    # NSSL histogram (outlined)
    label = f'NSSL'#f'{ModelData_NSSL.region}_{ModelData_NSSL.case}_{ModelData_NSSL.mpType}'
    axis.hist(valid_NSSL, bins=binEdges,
              histtype='step', linewidth=1.8, color='blue', label=label)

    # PRECIP CAMPAIGN PRECIPITATION DATA (outlined) 
    label = f'PRECIP QPE'# 01H'
    axis.hist(valid_NOAA, bins=binEdges,
              histtype='step', linewidth=1.8, color='black', label=label)

    # TEMPO histogram (outlined)
    label = f'TEMPO'#f'{ModelData_TEMPO.region}_{ModelData_TEMPO.case}_{ModelData_TEMPO.mpType}'
    axis.hist(valid_TEMPO, bins=binEdges,
              histtype='step', linewidth=1.8, color='green', label=label)

    # Log scale, labels, and limits
    axis.set_yscale("log")
    axis.set_xlim(left=0, right=np.nanmax(combinedData))
    axis.set_xlabel('Rainfall (mm)', fontsize=fontSettings["labelFont"])
    axis.set_ylabel('count', fontsize=fontSettings["labelFont"])

    # Add legend
    axis.legend(frameon=False, fontsize=fontSettings["legendFont"])

    # Fix XAxis Ticks
    maxRain = np.nanmax(combinedData)
    xMax = int(50 * np.ceil(maxRain / 50))
    axis.set_xlim(0, xMax)
    xticks = np.arange(0, xMax + 1, 50)
    axis.set_xticks(xticks)

    axis.tick_params(axis="both", labelsize=fontSettings["tickFont"])

In [ ]:
def SaveFigure(fig, ModelData1, ModelData2, dpi=300, plotType="combined"):
    """
    Saves a figure to the appropriate directory based on the models in combinedDict.
    """
    # --- Define output subdirectory and file path ---
    outputSubDirectory = f"{ModelData1.region}_{ModelData1.case}_{ModelData1.spinup_hours}hrs"
    os.makedirs(os.path.join(outputPlottingDirectory, outputSubDirectory), exist_ok=True)

    outputFilePath = os.path.join(
        outputPlottingDirectory,
        outputSubDirectory,
        f"RainfallHistogram_{plotType}.jpg"
    )

    # # --- Save figure ---
    # FigurePlotting_Class.SaveUniformFigure(fig, outputFilePath)
    fig.savefig(outputFilePath, dpi=dpi, bbox_inches="tight",pad_inches=0.02)
    plt.close(fig)
    print(f"Saved image: {outputFilePath}")

In [ ]:
####################################
#PLOTTING

import matplotlib as mpl
mpl.rcParams['figure.dpi'] = 300

In [ ]:
fontSettings = {
    "tickFont": 16+12,
    "labelFont": 18+12,
    "legendFont": 14+2,
    "titleFont": 22+6,
}

fig, axes, cax = CreateFigure_3x1(figsize=(25,20))

PlotRainContours(axes, cax, rain_NSSL_interp_whited, rain_TEMPO_interp_whited, precipData_T_Subset.latitude.values, precipData_T_Subset.longitude.values, clim,
                 ModelData_NSSL, ModelData_TEMPO,
                 precipData_T_Subset_whited)

fig.suptitle(
    f"{ModelData_NSSL.region} {ModelData_NSSL.case}",
    fontsize=fontSettings["titleFont"]+4,
    y=axes[0][1].get_position().y1+0.04,
    fontweight="bold"
)
SaveFigure(fig, ModelData1=ModelData_NSSL, ModelData2=ModelData_TEMPO, plotType="contour")

In [ ]:
fontSettings = {
    "tickFont": 16+2,
    "labelFont": 18+2,
    "legendFont": 14+2,
    "titleFont": 22,
}

fig, axes = CreateFigure_1x1(figsize=(8,4))

PlotRainHistograms(axes, rain_NSSL_interp, rain_TEMPO_interp, precipData_T_Subset,
                   axis=axes[0][0])

fig.suptitle(
    f"{ModelData_NSSL.region} {ModelData_NSSL.case}",
    fontsize=fontSettings["titleFont"],
    y=1.02,
    fontweight="bold"
)
SaveFigure(fig, ModelData1=ModelData_NSSL, ModelData2=ModelData_TEMPO, plotType="histogram")